In [1]:
import os
from dotenv import load_dotenv
load_dotenv()


os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [3]:
### create the data points 
from langsmith import Client
client = Client()

dataset_name = "Simple Chatbots  Evaluation"
dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {
                "answer": "A platform for observing and evaluating LLM applications"
            },
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {
                "answer": "A company that creates Large Language Models"
            },
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {
                "answer": "A technology company known for search"
            },
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {
                "answer": "A company that creates Large Language Models"
            },
        },
    ],

)

{'example_ids': ['dd6c60f3-d15b-48e7-86a5-1e14f6997176',
  'b106a39f-1938-4309-ab45-0b593558a632',
  '6f3d7ed6-8e05-40f4-9987-fabd1c95e7e8',
  '02d5eb8e-156c-42fa-831d-ab56ad567161',
  '35f67b58-381a-4edb-80b9-86c98315f772'],
 'count': 5,
 'as_of': '2026-08-02T05:18:08.991573935Z'}

### define metrcis (LLM as judge)

In [14]:
import openai
from langsmith import wrappers

openai_client = wrappers.wrap_openai(openai.OpenAI())
eval_instructions ="You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content},
        ],
    ).choices[0].message.content

    grade = response.strip().upper()
    if "INCORRECT" in grade:
        return False
    return "CORRECT" in grade

In [8]:
## Concisions - chekcs whether the actual output is less than 2X the length of th eexpected result.
def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(
        len(outputs["response"]) < 2 * len(reference_outputs["answer"])
    )

### Run evaluation 

In [10]:
default_instructions = (
    "Respond to the users question in a short, concise manner (one short sentence)."
)

def my_app(question: str, model: str = "gpt-4o-mini", instructions: str = default_instructions) -> str:
    return openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [15]:
# Wrapper function that maps dataset inputs to app outputs
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

# Evaluate with GPT-4o-mini
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="openai-4o-mini-chatbot",
)

View the evaluation results for experiment: 'openai-4o-mini-chatbot-140aeb24' at:
https://smith.langchain.com/o/64f94163-b9f9-4109-b8a4-da66be071676/datasets/ff7f6f18-e644-46d6-8c12-40db104e9c87/compare?selectedSessions=c3fe33c0-1f53-478b-8e48-d32efd3ef78e




5it [00:07,  1.44s/it]


In [16]:
# Wrapper for GPT-4-turbo
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"], model="gpt-4-turbo")}

# Evaluate with GPT-4-turbo
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="openai-4-turbo-chatbot",
)

View the evaluation results for experiment: 'openai-4-turbo-chatbot-a4e34b60' at:
https://smith.langchain.com/o/64f94163-b9f9-4109-b8a4-da66be071676/datasets/ff7f6f18-e644-46d6-8c12-40db104e9c87/compare?selectedSessions=310acf07-7193-47db-bc4a-a43ccc58c61a




5it [00:14,  2.81s/it]
